# Домашнее задание 1

In [ ]:
import json
import re
from collections import Counter
from collections.abc import Callable, Collection, Iterable, Mapping
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from statistics import mean
from typing import Any, Self, cast

import pandas as pd
from openai import LengthFinishReasonError, OpenAI
from openai.types.chat import ChatCompletion, ChatCompletionMessageParam
from pydantic import BaseModel, ValidationError, field_validator, model_validator

In [ ]:
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    timeout=3600,
)
models = client.models.list()
MODEL = models.data[0].id
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}
SAMPLING: dict[str, Any] = {
    "temperature": 0.7,
    "top_p": 0.8,
    "extra_body": {"top_k": 20},
}
RULES = (
    "По правилам скидка распределяется пропорционально стоимости товаров, доставка не является товаром, "
    "деньги за доставку не возвращаются."
)
ORDER = (
    "Наушники стоят 4000 руб., чехол 1000 руб. Скидка на товары 1000 руб., доставка 300 руб. "
    "Возвращаю только наушники."
)
HEADPHONES = {
    "input": ORDER,
    "answer": 3200,
    "returned_rub": 4000,
    "discount_rub": 1000,
    "delivery_rub": 300,
}
pd.set_option("display.max_colwidth", 250)

Message = Mapping[str, Any]


def chat(
    messages: Iterable[Message],
    think: bool = False,
    extra_body: dict[str, Any] | None = None,
    **params: Any,
) -> ChatCompletion:
    """Ответ модели целиком, вместе с usage; по умолчанию жадный выбор без рассуждения."""
    params = {
        "temperature": 0,
        "max_tokens": 1000,
        **params,
    }
    base = {} if think else NO_THINK
    extra_body = base | (extra_body or {})

    return client.chat.completions.create(
        model=MODEL,
        messages=cast(Iterable[ChatCompletionMessageParam], messages),
        extra_body=extra_body,
        **params,
    )


def structured[Schema: BaseModel](
    messages: Iterable[Message],
    schema: type[Schema],
    extra_body: dict[str, Any] | None = None,
    **params: Any,
) -> tuple[Schema | None, int]:
    """Ответ по схеме без рассуждения и число токенов; None, если генерация оборвалась на max_tokens."""
    params = {
        "temperature": 0,
        "max_tokens": 1000,
        **params,
    }
    try:
        response = client.chat.completions.parse(
            model=MODEL,
            messages=cast(Iterable[ChatCompletionMessageParam], messages),
            response_format=schema,
            extra_body=NO_THINK | (extra_body or {}),
            **params,
        )
    except LengthFinishReasonError as error:
        usage = error.completion.usage
        assert usage is not None

        return None, usage.completion_tokens

    assert response.usage is not None

    return response.choices[0].message.parsed, response.usage.completion_tokens


def parallel[Item, Result](fn: Callable[[Item], Result], items: Collection[Item]) -> list[Result]:
    """fn для каждого элемента; запросы идут параллельно."""
    with ThreadPoolExecutor(max(1, len(items))) as pool:
        results = pool.map(fn, items)

        return list(results)


def refund(instruction: str, order: str) -> list[dict[str, str]]:
    """Запрос на расчёт: в system -- инструкция и правила компании, в user -- заказ."""
    return [
        {
            "role": "system",
            "content": instruction + "\n\n" + RULES,
        },
        {
            "role": "user",
            "content": order,
        },
    ]


def read_jsonl(path: str) -> list[dict[str, Any]]:
    """Записи JSONL-файла."""
    lines = Path(path).read_text(encoding="utf-8").splitlines()

    return [json.loads(line) for line in lines]

## Задача 1 (10 баллов)

Реализуйте Tree of Thoughts для задачи рассчета суммы возврата. Рассуждение должно идти по шагам плана `PLAN`: на каждом шаге модель должна предложить несколько вариантов, оценщик оценить их на корректность и способность привести к решению задачи. В случае некорректности или если рассуждение зашло не туда -- вариант отбрасывается.

Критерии:

- +3 балла: на каждый шаг генерируется n различных вариантов;
- +3 балла: Оценщик делает не более двух ошибок для готовых `STATES`;
- +4 балла: Полная реализация алгоритма с генерацией вариантов для каждого шага, их оценкой и продолжением корректных шагов. 

In [ ]:
PLAN = ["Стоимость товаров", "Скидка на возвращаемые товары", "Сумма возврата"]
GENERATOR = (
    'Вычисли только величину "{name}" по данным заказа и уже принятым шагам. Ответь одной строкой: '
    '"{name}: <вычисление> = <число> руб."'
)
EVALUATOR = (
    "Проверь частичное решение по правилам возврата и данным заказа. Укажи ошибку или подтверди, что этот "
    "вариант можно продолжать. Ответь кратко, в двух-трёх предложениях. Последней строкой напиши "
    '"Вердикт: продолжать" или "Вердикт: отбросить".'
)

In [ ]:
def propose(task: dict[str, Any], steps: list[str], n: int = 3) -> list[str]:
    """Предлагает не более n вариантов шага."""
    # Ваш код здесь

    return []


propose(HEADPHONES, [])

In [ ]:
def evaluate(task: dict[str, Any], steps: list[str]) -> bool:
    """Оценка частичного решения."""
    # Ваш код здесь

    return False


GOODS = "Стоимость товаров: 4000 + 1000 = 5000 руб."
SHARE = "Скидка на возвращаемые товары: 1000 · 4000 / 5000 = 800 руб."

# частичное решение заказа с наушниками и вердикт по правилам
STATES = [  
    ([GOODS], True),
    (["Стоимость товаров: 4000 + 1000 + 300 = 5300 руб."], False),
    ([GOODS, SHARE], True),
    ([GOODS, "Скидка на возвращаемые товары: 1000 / 2 = 500 руб."], False),
    ([GOODS, SHARE, "Сумма возврата: 4000 − 800 = 3200 руб."], True),
    ([GOODS, SHARE, "Сумма возврата: 4000 + 800 = 4800 руб."], False),
    ([GOODS, "Сумма возврата: 4000 руб."], False),
]
verdicts = parallel(lambda steps: evaluate(HEADPHONES, steps), [steps for steps, _ in STATES])

pd.DataFrame({
    "последний шаг": [steps[-1] for steps, _ in STATES],
    "по правилам": [verdict for _, verdict in STATES],
    "оценщик": verdicts,
})

In [ ]:
def tree_of_thoughts(task: dict[str, Any], n: int = 3) -> list[str] | None:
    """Возвращает первое найденное итоговое решение задачи, которое оценщик посчитал корректным."""
    # Ваш код здесь

    return None


tree_of_thoughts(HEADPHONES)

## Задача 2 (5 баллов)

Необходимо реализовать валидацию ответа LLM:

- номер заказа -- допустимы только цифры;
- цена, скидка и сумма возврата не отрицательные целые числа;
- сумма возврата не больше цены товара.

В `refund_requests.jsonl` -- обращения и ответы на них. Все ответы подходят под схему, но часть нарушает правила валидации. Напишите валидаторы, а потом промпт `EXTRACTOR`, с которым модель отвечает по правилам.

Критерии:

- +3 балла: по одному баллу за каждый валидатор
- +2 балла: доля корректных ответов модели не меньше 0.9 (+1 балл если меньше.).

In [ ]:
class RefundRequest(BaseModel):
    order_id: str
    item: str
    price_rub: int
    discount_rub: int
    refund_rub: int

    @field_validator("order_id")
    @classmethod
    def order_id_digits(cls, value: str) -> str:
        """Номер заказа состоит только из цифр."""
        # Ваш код здесь

        return value

    @field_validator("price_rub", "discount_rub", "refund_rub")
    @classmethod
    def not_negative(cls, value: int) -> int:
        """Цена, скидка и сумма возврата не отрицательные."""
        # Ваш код здесь

        return value

    @model_validator(mode="after")
    def refund_within_price(self) -> Self:
        """Сумма возврата не больше цены."""
        # Ваш код здесь

        return self

In [ ]:
def is_valid(text: str) -> bool:
    """Проходит ли JSON-ответ проверку RefundRequest."""
    try:
        RefundRequest.model_validate_json(text)
    except ValidationError:
        return False

    return True


cases = read_jsonl("../data/refund_requests.jsonl")
case_answers = [json.dumps(case["answer"], ensure_ascii=False) for case in cases]

pd.DataFrame({
    "обращение": [case["input"] for case in cases],
    "ответ": case_answers,
    "принят": [is_valid(answer) for answer in case_answers],
})

In [ ]:
NAIVE = "Извлеки из обращения номер заказа, товар, его цену, скидку на него и сумму, которую нужно вернуть клиенту."
EXTRACTOR = NAIVE  # Ваш код здесь


def extract(text: str, prompt: str) -> RefundRequest | None:
    """Ответ модели по схеме RefundRequest; None, если он нарушает правила."""
    messages = [
        {
            "role": "system",
            "content": prompt,
        },
        {
            "role": "user",
            "content": text,
        },
    ]
    try:
        answer, _ = structured(messages, RefundRequest, max_tokens=200)
    except ValidationError:
        return None

    return answer


PROMPTS = {
    "простой промпт": NAIVE,
    "свой промпт": EXTRACTOR,
}
extraction: dict[str, list[Any]] = {"обращение": [case["input"] for case in cases]}
for name, prompt in PROMPTS.items():
    extracted = parallel(lambda case: extract(case["input"], prompt), cases)
    accepted = [answer is not None for answer in extracted]
    extraction[name] = extracted
    print(f"{name}: доля принятых {mean(accepted):.1f}")

pd.DataFrame(extraction)

## Задача 3 (5 баллов)

Необходимо решить задачу расчёта суммы возврата через Structured Outputs. Сумма должна возвращаться в поле `refund_rub`.

Напишите схему `Solution` и промпт `SOLVER`: перед тем как заполнить `refund_rub` модель должна выписать дополнительные вычисления. Учтите, что поля модель генерирует по порядку.

Постарайтесь решить без включения Reasoning.

Критерии:

- +1 балл: для всех кейсов модель генерирует валидный ответ
- +1 балл: кроме поля `refund_rub` -- в схеме есть ещё поля для промежуточных вычислений;
- +1 балл: в промпте нет правильных ответов из `TASKS`.
- +1 балл: доля правильных ответов больше 0.5
- +1 балл: Не использован Reasoning

In [ ]:
TASKS = read_jsonl("../data/refund_orders.jsonl")

pd.DataFrame(TASKS)[["input", "answer"]]

In [ ]:
class Solution(BaseModel):
    # Ваш код здесь
    pass


SOLVER = ""  # Ваш код здесь

In [ ]:
def solve_structured[Schema: BaseModel](
    task: dict[str, Any],
    schema: type[Schema],
    instruction: str,
    **params: Any,
) -> tuple[Schema | None, int]:
    """Решение по схеме и число токенов; None, если ответ не разобран."""
    messages = refund(instruction, task["input"])

    return structured(messages, schema, **params)


def refund_of(answer: BaseModel | None) -> int | None:
    """Сумма возврата из разобранного ответа."""
    return getattr(answer, "refund_rub", None)


solutions = parallel(lambda task: solve_structured(task, Solution, SOLVER), TASKS)
refunds = [refund_of(answer) for answer, _ in solutions]
unparsed = sum(answer is None for answer, _ in solutions)
right = sum(refund == task["answer"] for refund, task in zip(refunds, TASKS))
leaked = [task["answer"] for task in TASKS if re.search(rf"(?<!\d){task['answer']}(?!\d)", SOLVER)]
print("поля Solution:", " -> ".join(Solution.model_fields))
print(f"не разобрано {unparsed}, доля верных {right / len(TASKS):.1f}")
print("правильные ответы в промпте:", leaked or "нет")

pd.DataFrame({
    "задача": [task["input"] for task in TASKS],
    "эталон": [task["answer"] for task in TASKS],
    "ответ": refunds,
})

## Задача 4 (5 баллов)

Необходимо сравнить решение задачи 3 с базовым вариантом `Baseline`, в котором схема состоит только из поля `refund_rub`.

Напишите функцию `diagnose`, которая по сумме возврата и данным заказа определяет тип ответа:

- "верно" -- сумма совпала с правильной;
- "скидка вычтена целиком" -- из стоимости возвращаемых товаров вычтена вся скидка заказа;
- "скидка не вычтена" -- возвращена полная стоимость возвращаемых товаров;
- "доставка в сумме" -- к правильной сумме прибавлена доставка;
- "нет ответа" -- ответ не разобран;
- "другое" -- остальные ошибки.

Затем запустите оба варианта на `TASKS` с жадным выбором и с параметрами из `RUNS` и соберите таблицу: доля правильных ответов при жадном выборе, среднее, минимум и максимум доли по `RUNS`, токены на задачу и число ответов каждого типа.

Критерии:

- +3 балла: `diagnose` верно определяет тип всех ответов из `TESTS`;
- +2 балла: таблица сравнения для обоих вариантов;

In [ ]:
def diagnose(task: dict[str, Any], answer: int | None) -> str:
    """Тип ответа по сумме возврата."""
    # Ваш код здесь

    return "другое"


TESTS = [  # ответ на заказ с наушниками и его тип
    (3200, "верно"),
    (3000, "скидка вычтена целиком"),
    (4000, "скидка не вычтена"),
    (3500, "доставка в сумме"),
    (None, "нет ответа"),
    (2500, "другое"),
]

pd.DataFrame({
    "ответ": [answer for answer, _ in TESTS],
    "ожидается": [kind for _, kind in TESTS],
    "diagnose": [diagnose(HEADPHONES, answer) for answer, _ in TESTS],
})

In [ ]:
class Baseline(BaseModel):
    refund_rub: int


BASELINE = "Рассчитай сумму возврата за возвращаемые товары в рублях."
RUNS = {
    "seed 0": {**SAMPLING, "seed": 0},
    "seed 1": {**SAMPLING, "seed": 1},
    "seed 2": {**SAMPLING, "seed": 2},
}

# Ваш код здесь